# 06 · Four-Domain End-to-End Consulting Demo

Fully runnable offline demonstration using **synthetic**, explicitly fictional corpus. Re-run with course documents in notebook 09 to produce assignment-grounded evidence.

In [ ]:
%run 05_evaluation.ipynb

In [ ]:
from IPython.display import display, Markdown
seeded=seed_synthetic()
print('Seeded',len(seeded),'synthetic documents across',len(DOMAINS),'domains')
for domain,question,filename in CASES:
    matches=hybrid_search(PROJECT_ID,question,domain,top_k=2)
    print('\nDOMAIN:',domain.upper(),'QUESTION:',question)
    for x in matches:
        print(' -',x['citation'],'score=',x['score'],'excerpt=',x['excerpt'][:160])

In [ ]:
print('TECHNOLOGY — Safe recommendation with confirmed demo model')
result=rank_candidates({'project_id':PROJECT_ID,'domain':'technology',
  'query':'Which uplink module should we order after verifying CRC errors on DemoRouter-X revision B?',
  'device_model':'DemoRouter-X revision B',
  'candidates':[
   {'candidate_id':'demo-sfp10','name':'DemoSFP-10G','kind':'replacement_part',
    'compatible_models':['DemoRouter-X revision B'],'required_terms':['optical']},
   {'candidate_id':'wrong-module','name':'ZFiber-1G','kind':'replacement_part',
    'compatible_models':['DemoRouter-Z revision A']} ]})
print(json.dumps(result,indent=2)[:3500])

print('\nCROSS-DOMAIN — Technology + SEC evidence for joint manager-agent reasoning')
question='Compare optical router operational risk and supplier concentration business risk'
multi={d:hybrid_search(PROJECT_ID,question,d,top_k=2) for d in ['technology','sec']}
for d,records in multi.items():
    print(d,':',[x['citation'] for x in records])
print('Group 1 Manager Agent combines the evidence; Group 2 does not invent cross-domain decisions.')

In [ ]:
print('FINANCIAL — Reproducible synthetic dataset')
financials=pd.read_csv(ROOT/'data'/'synthetic'/'examplecorp_financials.csv')
finance=financial_metrics(financials,PROJECT_ID,'SYNTHETIC ExampleCorp dataset')
print(json.dumps({'cagr_pct':finance['cagr_pct'],'latest_margin_pct':finance['latest_margin_pct'],
                  'analysis_id':finance['analysis_id'],'rows':finance['rows']},indent=2))
print('Illustrative scenarios:',revenue_scenarios(finance['rows'][-1]['revenue'],finance['latest_margin_pct']))

metrics=evaluate_fixture()
print('\nSYNTHETIC EVALUATION ONLY')
print(json.dumps({k:v for k,v in metrics.items() if k!='details'},indent=2))
assert metrics['Recall@K']>=0.8, 'Synthetic recall regression'
assert metrics['citation_validity_rate']==1.0
assert all(test_recommendation_safety().values())
print('End-to-end demonstration checks passed')

In [ ]:
print('RESEARCH AUTOMATION — explicitly synthetic fixture mode')
research=run_research_task(PROJECT_ID,'csr','How can solar-powered cold storage reduce post-harvest food spoilage?',
                           source_mode='synthetic_fixture',top_k=3)
print(json.dumps({'status':research['status'],'source_mode':research['source_mode'],
                  'findings':[{'citation':x['citation'],'evidence_id':x['evidence_id']} for x in research['findings']]},indent=2))
assert research['status']=='completed' and research['findings']
assert retrieve_research_task(PROJECT_ID,research['research_task_id'])['status']=='completed'
print('Research batch and persistent evidence checks passed')